# Klarnet ses modeli (DDSP) · pilot · tek hücre

**Yapacağın üç şey:**
1. Üst menü: **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU → Kaydet**.
2. Aşağıdaki hücrenin solundaki **▶** düğmesine bas.
3. Açılan pencerede Google hesabını seçip **Drive'a izin ver**. **ORTAM HAZIR** yazısını görünce (2–3 dk) kalkabilirsin.

Gerisi kendiliğinden: veri, kısa deneme, eğitim, değerlendirme ve kör dinleme klasörü. Bitince çalışma zamanı kapanır.

- Bağlantı koparsa `DEVAM = 'son'` yapıp yeniden ▶: eğitim son kontrol noktasından sürer.
- Sonuç Drive'da: `ses_nota/ses-modeli/runs/<tarih>-klarnet-v1/sonuc_adim-…/kor-dinleme/`. Her klipte A ile B'den biri gerçek kayıt, öteki model. Anahtara bakmadan dinle, tahminini `cevap.csv`'ye yaz.

**İÇ TEST.** Veri, model ve sesler yalnız Drive'da kalır; paylaşılmaz.

In [ ]:
# Klarnet ses modeli (DDSP) pilotu -- TEK HÜCRE. A100.
# İç kayıtlardan hazırlanmış klarnet verisiyle bir tını modeli eğitir (acids-ircam/ddsp_pytorch, sabit sürüm).
# Perdeyi model seçmez, biz veririz; model yalnız tınıyı (harmonikler, üfleme gürültüsü, oda) öğrenir. Sonunda eğitimde
# hiç görülmemiş kayıtlardan klipler modelle yeniden çalınır: gerçek ile model yan yana ve karışık (A/B) kör dinleme.
# Veri: ses_nota/ses-modeli/<VERI>/ · kod: ses_nota/ses-modeli/kod.zip · çıktı: ses_nota/ses-modeli/runs/<tarih>-<AD>/
AD = 'klarnet-v1'     # koşu adı: runs/<tarih>-<AD>/ (aynı gün ikinci koşu -2, -3 ...; eski koşunun üstüne yazılmaz)
VERI = 'klarnet-v1'   # veri klasörü: ses_nota/ses-modeli/<VERI>/
ADIM = 30000          # eğitim adımı (önceki tanbur denemesi 30.000 adımda bitti: L4'te ~40 dk, 16 kHz)
DEVAM = None          # oturum kesilirse: 'son' (bu AD'li en son koşunun son kontrol noktasından sürer)
KAPAT = True          # bitince Drive'ı kaydedip çalışma zamanını kapat (kota harcamasın)
A100_SART = True      # A100 yoksa eğitime başlamadan dur · False: yine de koş (L4'te yaklaşık iki kat uzun sürer)
GELISMIS = ''         # boş bırak (yalnız model ayarı için, ör. 'GIZLI=256, PARTI=8')
# ----------------------------------------------------------------------------------------------------
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU yok'
print('GPU:', GPU)
if A100_SART and 'A100' not in GPU:
    raise SystemExit(f'A100 değil ({GPU}): Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU seç, yeniden çalıştır. '
                     'A100 olmadan koşmak istersen A100_SART = False yap.')
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, zipfile
DRIVE = '/content/drive/MyDrive/ses_nota'
SM = f'{DRIVE}/ses-modeli'
KOD = '/content/klarnet-kod'
shutil.rmtree(KOD, ignore_errors=True)
zipfile.ZipFile(f'{SM}/kod.zip').extractall(KOD)
sys.path.insert(0, KOD)
import ortak, klarnet_tek as kt
_kapat = KAPAT
A = None
try:
    A = kt.ayarlar(AD, ADIM, DEVAM, GELISMIS, kok=SM, veri_adi=VERI, gpu=GPU)
    kt.kurulum(A)                                           # torchcrepe, resampy, pyloudnorm; DDSP kodu sabit sürümden
    kt.surum_denetimi(A)                                    # kısa denemeler; 'ORTAM HAZIR' görününce kalkabilirsin
    VERI_ = ortak.tek_adim(A, 'veri', kt.veri)              # Drive'dan /content'e, dosya boyutu denetimiyle
    E = ortak.tek_adim(A, 'hazirlik', kt.hazirlik, VERI_)   # koşu klasörü; DEVAM ise son kontrol noktası
    ortak.tek_adim(A, 'mini', ortak.tek_mini, E)            # birkaç adım: hız, fp32/bf16, kontrol noktası yaz-oku
    ortak.tek_adim(A, 'egitim', ortak.tek_egit, E)          # ADIM'a kadar; 12 dk'da bir kontrol noktası Drive'a
    ortak.tek_adim(A, 'degerlendirme', kt.degerlendir, E)   # yeniden sentez, ölçüler, A/B kör dinleme klasörü
except KeyboardInterrupt:  # elle durduruldu: çalışma zamanı açık kalsın
    _kapat = False
    raise
finally:
    if A is not None:
        kt.ozet(A)
    print('BİTTİ.')
    if _kapat:
        print('Drive kaydediliyor, çalışma zamanı kapatılıyor (KAPAT = True)...')
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()
    else:
        print('Çalışma zamanı → Çalışma zamanının bağlantısını kes ve sil (A100 boşta da kota harcar).')